<a href="https://colab.research.google.com/github/osoliman/HTM737/blob/main/Week10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [ ]:
df_clinical_care = pd.read_csv('https://raw.githubusercontent.com/osoliman/Chapter6_HTM737/refs/heads/main/hvbp_clinical_care_11_07_2017.csv')
df_clinical_care.head()

In [ ]:
df_efficiency = pd.read_csv('https://raw.githubusercontent.com/osoliman/Chapter6_HTM737/refs/heads/main/hvbp_efficiency_11_07_2017.csv')
df_efficiency.head()

In [ ]:
df_hcahps = pd.read_csv('https://raw.githubusercontent.com/osoliman/Chapter6_HTM737/refs/heads/main/hvbp_hcahps_11_07_2017.csv')
df_hcahps.head()

In [ ]:
df_safety = pd.read_csv('https://raw.githubusercontent.com/osoliman/Chapter6_HTM737/refs/heads/main/hvbp_safety_11_07_2017.csv')
df_safety.head()

In [ ]:
df_tps = pd.read_csv('https://raw.githubusercontent.com/osoliman/Chapter6_HTM737/refs/heads/main/hvbp_tps_11_07_2017.csv')
df_tps.head()

In [2]:
pathname = 'https://raw.githubusercontent.com/osoliman/Chapter6_HTM737/refs/heads/main/'
files_of_interest = [
    'hvbp_tps_11_07_2017.csv',
    'hvbp_clinical_care_11_07_2017.csv',
    'hvbp_safety_11_07_2017.csv',
    'hvbp_efficiency_11_07_2017.csv',
    'hvbp_hcahps_11_07_2017.csv'
]

dfs = {
    foi: pd.read_csv(pathname + foi, header=0) for foi in files_of_interest
}

In [3]:
type(dfs)

dict

In [ ]:
dfs.items()

In [4]:
for k, v in dfs.items():
    print(
        f'{k} - Number of rows: {v.shape[0]}, Number of columns: {v.shape[1]}'
    )

hvbp_tps_11_07_2017.csv - Number of rows: 2808, Number of columns: 16
hvbp_clinical_care_11_07_2017.csv - Number of rows: 2808, Number of columns: 28
hvbp_safety_11_07_2017.csv - Number of rows: 2808, Number of columns: 64
hvbp_efficiency_11_07_2017.csv - Number of rows: 2808, Number of columns: 14
hvbp_hcahps_11_07_2017.csv - Number of rows: 2808, Number of columns: 73


In [5]:
for v in dfs.values():
    for column in v.columns:
        print(column)
    print()

Provider Number
Hospital Name
Address
City
State
Zip Code
County Name
Unweighted Normalized Clinical Care Domain Score
Weighted Normalized Clinical Care Domain Score
Unweighted Patient and Caregiver Centered Experience of Care/Care Coordination Domain Score
Weighted Patient and Caregiver Centered Experience of Care/Care Coordination Domain Score
Unweighted Normalized Safety Domain Score
Weighted Safety Domain Score
Unweighted Normalized Efficiency and Cost Reduction Domain Score
Weighted Efficiency and Cost Reduction Domain Score
Total Performance Score

Provider Number
Hospital Name
Address
City
State
ZIP Code
County Name
MORT-30-AMI Achievement Threshold
MORT-30-AMI Benchmark
MORT-30-AMI Baseline Rate
MORT-30-AMI Performance Rate
MORT-30-AMI Achievement Points
MORT-30-AMI Improvement Points
MORT-30-AMI Measure Score
MORT-30-HF Achievement Threshold
MORT-30-HF Benchmark
MORT-30-HF Baseline Rate
MORT-30-HF Performance Rate
MORT-30-HF Achievement Points
MORT-30-HF Improvement Points
MOR

In [6]:
#Unify the key across all datasets to be Provider Number (as one of them is Provider_Number)
for df in dfs.values():
    df.rename(columns={'Provider_Number': 'Provider Number'}, inplace=True)

#Optional you can run the command above to make sure that they all become Provider Number

In [7]:
#Joining the 5 files together on Provider Number
#Start with first file
df_master = dfs[files_of_interest[0]]

for foi in files_of_interest[1:]:
    df_master = df_master.merge(
        dfs[foi],
        on='Provider Number',
        how='left',
        copy=False
    )

print(df_master.shape)

MergeError: Passing 'suffixes' which cause duplicate columns {'State_x', 'City_x', 'Address_x'} is not allowed.

In [8]:
from collections import Counter

# Step 1: Collect all column names from all dataframes
all_columns = []
for df in dfs.values():
    all_columns.extend(df.columns)

# Step 2: Count how many times each column appears
column_counts = Counter(all_columns)

# Step 3: Find columns that appear more than once (excluding merge key)
common_cols = {col for col, count in column_counts.items()
               if count > 1 and col != 'Provider Number'}

print(f"Columns to drop: {common_cols}")

Columns to drop: {'City', 'Address', 'State', 'County Name', 'Hospital Name', 'ZIP Code'}


In [9]:
for col, count in column_counts.most_common():
    if count > 1:
        print(f"{col}: {count} times")

Provider Number: 5 times
Address: 5 times
City: 5 times
State: 5 times
Hospital Name: 4 times
County Name: 4 times
ZIP Code: 3 times


In [10]:
# Start with the first dataframe
df_master = dfs[files_of_interest[0]]

# Merge the rest, dropping duplicate columns
for foi in files_of_interest[1:]:
    df_to_merge = dfs[foi].drop(columns=common_cols, errors='ignore')
    df_master = df_master.merge(
        df_to_merge,
        on='Provider Number',
        how='left',
        copy=False
    )

print(df_master.shape)

(2808, 170)


In [11]:
#Writing csv
df_master.to_csv('hvbp.csv', index=False)
#Then check folders on the left

In [12]:
#New Python concept
#List comprehension example

#Remember this from loops?

def f_to_c(temp_f):
    return (temp_f - 32) * 5/9

temps_f = [32, 68, 98.6, 212]
temps_c = []
for temp in temps_f:
    temps_c.append(f_to_c(temp))

print(temps_c)  # [0.0, 20.0, 37.0, 100.0]

[0.0, 20.0, 37.0, 100.0]


In [13]:
def f_to_c(temp_f):
    return (temp_f - 32) * 5/9

temps_f = [32, 68, 98.6, 212]
temps_c = [f_to_c(temp) for temp in temps_f] #list comprehension

print(temps_c)  # [0.0, 20.0, 37.0, 100.0]

[0.0, 20.0, 37.0, 100.0]
